In [2]:
import requests
import pandas as pd
from datetime import datetime, timedelta

In [1]:
def inspect_iss_endpoint(url, params=None, max_rows=5):
    print("=" * 100)
    print("URL:", url)
    print("PARAMS:", params)

    response = requests.get(url, params=params)
    print("STATUS:", response.status_code)
    print("FINAL URL:", response.url)

    try:
        response.raise_for_status()
    except Exception as error:
        print("HTTP ERROR:", error)
        print(response.text[:500])
        return

    try:
        data = response.json()
    except Exception as error:
        print("JSON ERROR:", error)
        print(response.text[:500])
        return

    print("DATA KEYS:", data.keys())

    for key, value in data.items():
        print("-" * 100)
        print("TABLE:", key)

        if not isinstance(value, dict):
            print("Not a dict:", type(value))
            continue

        print("INNER KEYS:", value.keys())

        if "columns" not in value or "data" not in value:
            print("No columns/data in this table")
            continue

        columns = value["columns"]
        rows = value["data"]

        print("COLUMNS:")
        print(columns)

        df = pd.DataFrame(rows, columns=columns)

        print("SHAPE:", df.shape)

        if not df.empty:
            display(df.head(max_rows))
        else:
            print("EMPTY TABLE")

In [29]:
trade_date = "2026-05-22"

In [16]:
inspect_iss_endpoint(
    url="https://iss.moex.com/iss/turnovers.json",
    params={"date": trade_date}
)

IndentationError: unexpected indent (2372161011.py, line 2)

In [30]:
inspect_iss_endpoint(
    url="https://iss.moex.com/iss/engines/futures/turnovers.json",
    params={"date": trade_date}
)

URL: https://iss.moex.com/iss/engines/futures/turnovers.json
PARAMS: {'date': '2026-05-22'}
STATUS: 200
FINAL URL: https://iss.moex.com/iss/engines/futures/turnovers.json?date=2026-05-22
DATA KEYS: dict_keys(['turnovers', 'turnoversprevdate'])
----------------------------------------------------------------------------------------------------
TABLE: turnovers
INNER KEYS: dict_keys(['metadata', 'columns', 'data'])
COLUMNS:
['NAME', 'ID', 'VALTODAY', 'VALTODAY_USD', 'NUMTRADES', 'UPDATETIME', 'TITLE']
SHAPE: (3, 7)


,NAME,ID,VALTODAY,VALTODAY_USD,NUMTRADES,UPDATETIME,TITLE
0,forts,22.0,672610.932026,9501.469582,2073206,2026-05-22 23:49:59,Фьючерсы
1,options,24.0,9759.892906,137.870679,20051,2026-05-22 23:49:56,Опционы
2,TOTALS,NaN,682370.824932,9639.340261,2093257,2026-05-22 23:49:59,Всего по рынку


----------------------------------------------------------------------------------------------------
TABLE: turnoversprevdate
INNER KEYS: dict_keys(['metadata', 'columns', 'data'])
COLUMNS:
['NAME', 'ID', 'VALTODAY', 'VALTODAY_USD', 'NUMTRADES', 'UPDATETIME', 'TITLE']
SHAPE: (3, 7)


,NAME,ID,VALTODAY,VALTODAY_USD,NUMTRADES,UPDATETIME,TITLE
0,forts,22.0,712349.306929,10040.032007,2238797,2026-05-21 23:49:59,Фьючерсы
1,options,24.0,11633.100654,163.959874,24622,2026-05-21 23:49:58,Опционы
2,TOTALS,NaN,723982.407583,10203.991881,2263419,2026-05-21 23:49:59,Всего по рынку


In [8]:
inspect_iss_endpoint(
    url="https://iss.moex.com/iss/engines/futures/markets/forts/turnovers.json",
    params={"date": trade_date}
)

URL: https://iss.moex.com/iss/engines/futures/markets/forts/turnovers.json
PARAMS: None
STATUS: 200
FINAL URL: https://iss.moex.com/iss/engines/futures/markets/forts/turnovers.json
DATA KEYS: dict_keys(['turnovers'])
----------------------------------------------------------------------------------------------------
TABLE: turnovers
INNER KEYS: dict_keys(['metadata', 'columns', 'data'])
COLUMNS:
['NAME', 'ID', 'VALTODAY', 'VALTODAY_USD', 'NUMTRADES', 'UPDATETIME', 'TITLE', 'BOARDID']
SHAPE: (1, 8)


,NAME,ID,VALTODAY,VALTODAY_USD,NUMTRADES,UPDATETIME,TITLE,BOARDID
0,RFUD,101,6444.4589,90.2946,53298,2026-05-30 16:31:26,Фьючерсы,RFUD


In [4]:
# если перейти по ссылке, и посмотреть на архивные ссылки, окажется, что нужна подписка
url = "`https://iss.moex.com/iss/archives/engines/futures/markets/forts/securities/monthly.zip`"

data = requests.get(url).json()
print(len(data['files']['data']))
print(data.keys())
print(data['files'].keys())
print(data['files']['columns'])
df = pd.DataFrame(
    data["files"]["data"],
    columns=data["files"]["columns"]
)
df


InvalidSchema: No connection adapters were found for '`https://iss.moex.com/iss/archives/engines/futures/markets/forts/securities/monthly.zip`'

In [ ]:
def generate_dates(start_date, end_date):
    start_dt = datetime.strptime(start_date, "%Y-%m-%d").date()
    end_dt = datetime.strptime(end_date, "%Y-%m-%d").date()
    current_date = start_dt
    while current_date <= end_dt:
        yield current_date.isoformat()
        current_date += timedelta(days=1)

In [ ]:
# оказалось, пустышка1
url = "https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/Si/openpositions.json"
for date in generate_dates("2026-05-20", "2026-05-31"):
    params = {
        "date": date,
        "asset_type": "F",
        "option_type": "P"
    }
    response = requests.get(url, params=params)
    data = response.json()
    print(data.keys())
    df = pd.DataFrame(
        data['open_positions'],
        columns=data['open_positions']['columns']
    )
    if not df.empty:
        print(df)

In [ ]:
# оказалось, пустышка2
url1 = "https://iss.moex.com/iss/statistics/engines/futures/derivatives/expirationopenpositions.json"
date = "2026-05-20"
params = {
    "date": date
}
response = requests.get(url1, params=params)
data = response.json()
print(data)
